In [1]:
# standard library
import os
import re
import sys
import subprocess
import importlib.util

# third-party
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from helper.io import load_data
from helper.geno import per_genotype_freq_per_human, top_correlated_snps
from helper.pheno import make_pheno
from helper.plotting import show_biases
from helper.gwas import fit_line
from helper.effect_simulation import simulate_effects
from helper.pca import get_n_pcs
from helper.distance_to_snp import PC_distance_to_snp, pearson_distance_to_snp, marginal_estimation
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
raw_geno, humans = load_data()
snp_names = list(raw_geno.columns)
majaf, hetaf, minaf = per_genotype_freq_per_human(raw_geno, pop=humans["populations"])

In [3]:
pheno = make_pheno(humans)

In [4]:
median_snp = raw_geno.columns[len(raw_geno.columns) // 2]
top_corrs_with_median_snps = top_correlated_snps(raw_geno, median_snp, n=50)
least_corrs_with_median_snps = top_correlated_snps(raw_geno, median_snp, n=50, least=True)

In [8]:
import pandas as pd

def snp_effect_score(
    raw_geno: pd.DataFrame,
    snp_corrs: pd.Series,
) -> pd.Series:
    """
    Construct a per-sample effect score as a weighted sum of SNPs.

    Parameters
    ----------
    raw_geno : pd.DataFrame
        Samples x SNPs genotype matrix.
    snp_corrs : pd.Series
        Index = SNP names, values = weights (e.g., correlations).

    Returns
    -------
    pd.Series
        Effect score per sample.
    """
    snps = snp_corrs.index
    weights = snp_corrs.values

    return raw_geno[snps].values @ weights

In [9]:
effect_top = snp_effect_score(raw_geno, top_corrs_with_median_snps)
effect_least = snp_effect_score(raw_geno, least_corrs_with_median_snps)

In [10]:
effect_top

array([15.6196134 , 18.23336844, 16.30656381, ..., 19.61597741,
       19.37070237, 19.37556717])

In [11]:
effect_least

array([ 0.0021722 , -0.00049106,  0.00151169, ...,  0.00169017,
        0.00045531,  0.00115378])

In [12]:
top_corrs_with_median_snps

R_3588_MAF_0.137    0.571396
R_4055_MAF_0.115    0.568826
R_3950_MAF_0.118    0.510838
R_3411_MAF_0.148    0.504128
R_4213_MAF_0.108    0.501480
R_3739_MAF_0.128    0.491603
R_3103_MAF_0.165    0.477499
R_3010_MAF_0.170    0.471830
R_4997_MAF_0.087    0.466881
R_4582_MAF_0.097    0.457177
R_4078_MAF_0.114    0.446079
R_3564_MAF_0.138    0.445020
R_2814_MAF_0.183    0.441865
C_2483_MAF_0.208    0.435282
R_4738_MAF_0.093    0.432683
R_4694_MAF_0.095    0.432019
R_3204_MAF_0.160    0.430314
R_5228_MAF_0.081    0.429718
R_2745_MAF_0.188    0.426374
R_3343_MAF_0.152    0.426060
C_2459_MAF_0.211    0.425920
R_2618_MAF_0.198    0.424381
R_4267_MAF_0.106    0.422929
C_2398_MAF_0.216    0.422899
R_4899_MAF_0.089    0.420721
R_4786_MAF_0.092    0.420075
R_2817_MAF_0.183    0.416319
C_2386_MAF_0.217    0.416259
R_2619_MAF_0.198    0.414876
R_4088_MAF_0.113    0.414488
R_2768_MAF_0.186    0.414046
R_3254_MAF_0.157    0.413957
R_4758_MAF_0.093    0.411576
R_5327_MAF_0.079    0.407994
R_3409_MAF_0.1